## Install the recsys_slates_dataset pip package


In [1]:
!pip install "cython<3.0.0" && pip install --no-build-isolation pyyaml==5.4.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.7 MB/s eta 0:00:0000:010:01
  Attempting uninstall: cython
    Found existing installation: Cython 3.0.12
    Uninstalling Cython-3.0.12:
      Successfully uninstalled Cython-3.0.12
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.1/175.1 kB 4.1 MB/s eta 0:00:0000:01
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyyaml: filename=PyYAML-5.4.1-cp312-cp312-linux_x86_64.whl size=45657 sha256=46c77479fd3eafab216325dd1de85427976ed670a6aed4f872bfcc9b0369ed38
  Stored in directory: /root/.cache/pip/wheels/db/55/e5/815110eef60d46b744555e67a31f4925346437926cedac7065
Successfully built pyyaml
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 6.0.3
    Uninstalling PyYAML-6.0.3:
      Successfully uninstalled PyYAML-6.0.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
goog

In [2]:
!pip install recsys_slates_dataset -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
!cp -r /kaggle/input/datasets/ireneskvortsova/diploma-dataset /kaggle/working

In [ ]:
from torch.utils.data import Dataset, DataLoader
class SequentialDataset(Dataset):
    '''
     Note: displayType has been uncommented for future easy implementation.
    '''
    def __init__(self, data, sample_uniform_slate=False):

        self.data = data
        self.num_items = self.data['slate'].max()+1
        self.sample_uniform_slate = sample_uniform_slate
        print(f"Loading dataset with slate size={self.data['slate'].size()} and uniform candidate sampling={self.sample_uniform_slate}")

    def __getitem__(self, idx):
        batch = {key: val[idx] for key, val in self.data.items()}

        if self.sample_uniform_slate:
            action = torch.randint_like(batch['slate'], low=3, high=self.num_items)

            action[:,0] = 1
            clicked = batch['click']!=1
            action[:,1][clicked] = batch['click'][clicked]
            batch['slate'] = action
            batch['click_idx'] = clicked.long()

        return batch

    def __len__(self):
        return len(self.data['click'])

In [4]:
import numpy as np
import json
def load_dataloaders(data_dir= "dat",
                     batch_size=1024,
                     num_workers= 0,
                     sample_candidate_items=False,
                     valid_pct= 0.05,
                     test_pct= 0.05,
                     t_testsplit= 5,
                     limit_num_users=None,
                     seed=0):
    """
    Loads pytorch dataloaders to be used in training. If used with standard settings, the train/val/test split is equivalent to Eide et. al. 2021.

    Attributes:
      data_dir: [str] where download and store data if not already downloaded.
      batch_size: [int] Batch size given by dataloaders.
      num_workers: [int] How many threads should be used to prepare batches of data.
      sample_candidate_items: [int] Number of negative item examples sampled from the item universe for each interaction. If positive, the dataset provide an additional dictionary item "allitem". Often also called uniform candidate sampling. See Eide et. al. 2021 for more information.
      valid_pct: [float] Percentage of users allocated to validation dataset.
      test_pct: [float] Percentage of users allocated to test dataset.
      t_testsplit: [int] For users allocated to validation and test datasets, how many initial interactions should be part of the training dataset.
      limit_num_users: [int] For debugging purposes, only return some users.
      seed: [int] Seed used to sample users/items.

    """
    print('Load data..')
    with np.load("{}/data.npz".format(data_dir)) as data_np:
        data = {key: torch.tensor(val) for key, val in data_np.items()}
    print('Data loaded')

    if limit_num_users is not None:
        print("Limiting dataset to only return the first {} users.".format(limit_num_users))
        data = {key : val[:limit_num_users] for key, val in data.items()}

    with open('{}/ind2val.json'.format(data_dir), 'rb') as handle:
        # Use string2int object_hook found here: https://stackoverflow.com/a/54112705
        ind2val = json.load(
            handle,
            object_hook=lambda d: {
                int(k) if k.lstrip('-').isdigit() else k: v
                for k, v in d.items()
                }
            )

    num_users = len(data['click'])
    num_validusers = int(num_users * valid_pct)
    num_testusers = int(num_users * test_pct)
    torch.manual_seed(seed)
    perm_user = torch.randperm(num_users)
    valid_user_idx = perm_user[:num_validusers]
    test_user_idx  = perm_user[num_validusers:(num_validusers+num_testusers)]
    train_user_idx = perm_user[(num_validusers+num_testusers):]

    # Split dictionary into train/valid/test with a phase mask that shows which interactions are in different sets
    # (as some users have both train and valid data)
    data_train = data
    data_train['phase_mask'] = torch.ones_like(data['click']).bool()
    data_train['phase_mask'][test_user_idx,t_testsplit:]=False
    data_train['phase_mask'][valid_user_idx,t_testsplit:]=False

    data_valid = {key: val[valid_user_idx] for key, val in data.items()}
    data_valid['phase_mask'] = torch.zeros_like(data_valid['click']).bool()
    data_valid['phase_mask'][:,t_testsplit:] = True

    data_test = {key: val[test_user_idx] for key, val in data.items()}
    data_test['phase_mask'] = torch.zeros_like(data_test['click']).bool()
    data_test['phase_mask'][:,t_testsplit:] = True

    data_dicts = {
        "train" : data_train,
        "valid" : data_valid,
        "test" : data_test}

    datasets = {
        phase : SequentialDataset(data, sample_candidate_items)
        for phase, data in data_dicts.items()
        }


    # Build dataloaders for each data subset:
    dataloaders = {
        phase: DataLoader(ds, batch_size=batch_size, shuffle=(phase=="train"), num_workers=num_workers)
        for phase, ds in datasets.items()
    }
    for key, dl in dataloaders.items():
        print(
            "In {}: num_users: {}, num_batches: {}".format(key, len(dl.dataset), len(dl))
        )

    # Load item attributes:
    with np.load('{}/itemattr.npz'.format(data_dir), mmap_mode=None) as itemattr_file:
        itemattr = {key : val for key, val in itemattr_file.items()}

    return ind2val, itemattr, dataloaders

In [5]:
import torch
# from recsys_slates_dataset import dataset_torch
ind2val, itemattr, dataloaders = load_dataloaders(data_dir="/kaggle/working/diploma-dataset/")

print("Dictionary containing the dataloaders:")
print(dataloaders)

Load data..
Data loaded
Loading dataset with slate size=torch.Size([2277645, 20, 25]) and uniform candidate sampling=False
Loading dataset with slate size=torch.Size([113882, 20, 25]) and uniform candidate sampling=False
Loading dataset with slate size=torch.Size([113882, 20, 25]) and uniform candidate sampling=False
In train: num_users: 2277645, num_batches: 2225
In valid: num_users: 113882, num_batches: 112
In test: num_users: 113882, num_batches: 112
Dictionary containing the dataloaders:
{'train': <torch.utils.data.dataloader.DataLoader object at 0x7f2fafcfabd0>, 'valid': <torch.utils.data.dataloader.DataLoader object at 0x7f2fafcfb380>, 'test': <torch.utils.data.dataloader.DataLoader object at 0x7f2f78818410>}


In [7]:
# Load interaction data
dat = dataloaders['train'].dataset.data

# Print dimensions of all arrays:
for key, val in dat.items():
  print(f"{key} : \t {val.size()}")

userId : 	 torch.Size([2277645])
click : 	 torch.Size([2277645, 20])
click_idx : 	 torch.Size([2277645, 20])
slate_lengths : 	 torch.Size([2277645, 20])
slate : 	 torch.Size([2277645, 20, 25])
interaction_type : 	 torch.Size([2277645, 20])
phase_mask : 	 torch.Size([2277645, 20])


In [11]:
print("Slate:")
print(dat['slate'][5,3])
print(" ")
print("Click:")
print(dat['click'][5,3])
print("Type of interaction: (1 implies search, see ind2val file)")
print(dat['interaction_type'][5,3])

Slate:
tensor([     1, 638995, 638947, 638711, 637590, 637930, 638894,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0])
 
Click:
tensor(637590)
Type of interaction: (1 implies search, see ind2val file)
tensor(1)


In [12]:
print("Click_idx:")
print(dat['click_idx'][5,3])
print("Slate lengths:")
print(dat['slate_lengths'][5,3])

Click_idx:
tensor(4)
Slate lengths:
tensor(7)


## PPO experiment

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.distributions import Categorical
import numpy as np
import random
from tqdm import tqdm

In [58]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
class RewardModelDataset(Dataset):
      def __init__(self, dat, item2id, item_cat_tensor, max_len=25, n_users=None):
          self.samples = []
          self.max_len = max_len
          self.item_cat = item_cat_tensor

          clicks = dat['click'] # [n_users, 20]
          slates = dat['slate'] # [n_users, 20, slate_len]
          # lengths = dat['slate_lengths'] # [n_usesrs, 20]
          n = n_users or clicks.shape[0]

          for u in tqdm(range(n), desc="Building RewardModel dataset"):
              for t in range(clicks.shape[1]):
                  click_item = clicks[u, t].item()
                  if click_item < 3 or click_item not in item2id:
                      continue
                  slate_raw = slates[u, t]
                  last_item_index = len(slate_raw) - 1
                  while slate_raw[last_item_index] <= 3 and last_item_index >= 0:
                      if last_item_index <= 0:
                          print(u, t, last_item_index, len(slate_raw))
                      last_item_index -= 1

                  history = []
                  history_cats = []
                  for i in range(last_item_index + 1):
                      item_orig = slate_raw[i].item()
                      if item_orig <= 3 or item_orig not in item2id:
                          continue

                      item_id = item2id[item_orig]
                      label = 1.0 if i == last_item_index else 0.0
                      history_cats.append(item_cat_tensor[item_id].item())
                      if len(history) > 2:
                          self.samples.append((history.copy(), history_cats.copy(), item_id, label))
                      history.append(item_id)

      def pad(self, seq):
          res = torch.zeros(self.max_len, dtype=torch.long)
          n = min(len(seq), self.max_len)
          if n > 0:
              res[-n:] = torch.tensor(seq[-n:], dtype=torch.long)
          return res

      def __len__(self):
          return len(self.samples)

      def __getitem__(self, idx):
          history, history_cats, item_id, label = self.samples[idx]
          return {
              'state': self.pad(history),
              'item': torch.tensor(item_id, dtype=torch.long),
              'category': history_cats[-1],
              'category_history': self.pad(history_cats),
              'label': torch.tensor(label, dtype=torch.float32),
          }

In [ ]:
class RewardModel(nn.Module):
    """Предсказывает P(user leaves | state, item)."""
    def __init__(self, item_dim, num_categories, hidden_dim, item_cat_tensor, padding_idx=0):
        super().__init__()
        self.padding_idx = padding_idx
        self.item_emb = nn.Embedding(item_dim + 1, hidden_dim, padding_idx=padding_idx)
        self.cat_emb = nn.Embedding(num_categories + 2, hidden_dim // 4, padding_idx=0)
        self.encoder_items = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        self.encoder_cats = nn.GRU(hidden_dim // 4, hidden_dim // 4, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim + hidden_dim + hidden_dim // 4, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.register_buffer('item_categories', item_cat_tensor)

    def get_state_repr(self, states_seq):
        embeds = self.item_emb(states_seq) # [B, L, H]
        _, h_n = self.encoder_items(embeds)
        return h_n.squeeze(0) # [B, H]

    def get_category_repr(self, states_seq):
        embeds = self.cat_emb(states_seq) # [B, L, H // 4]
        _, h_n = self.encoder_cats(embeds)
        return h_n.squeeze(0)

    def forward(self, state_seq, item_id, category_history):
        s = self.get_state_repr(state_seq) # [B, H]
        a = self.item_emb(item_id) # [B, H]
        cat = self.get_category_repr(category_history) # [B, H/4]
        return self.head(torch.cat([s, a, cat], dim=-1)).squeeze(-1)

    @torch.no_grad()
    def p_continue(self, state_seq, slate_item_categories, item_id):
        return 1.0 - torch.sigmoid(self.forward(state_seq, item_id, slate_item_categories))

In [ ]:
def save_reward_model(reward_model, item2id, item_dim, num_categories, hidden_dim, path):
    torch.save({
        'state_dict': reward_model.state_dict(),
        'item2id': item2id,
        'item_dim': item_dim,
        'num_categories': num_categories,
        'hidden_dim': hidden_dim,
    }, path)
    print(f"RewardModel saved to {path}")


def load_reward_model(path, device):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    item2id = ckpt['item2id']
    item_dim = ckpt['item_dim']
    num_categories = ckpt['num_categories']
    hidden_dim = ckpt['hidden_dim']

    item_cat_tensor = ckpt['state_dict']['item_categories'].cpu()

    model = RewardModel(item_dim = item_dim, num_categories  = num_categories,
        hidden_dim = hidden_dim, item_cat_tensor = item_cat_tensor).to(device)
    model.load_state_dict(ckpt['state_dict'])
    model.eval()
    print(f"RewardModel loaded from {path}  (item_dim={item_dim}, hidden={hidden_dim})")
    return model, item2id


def save_ppo_agent(ppo_agent, item2id, item_dim, hidden_dim, path):
    torch.save({
        'state_dict': ppo_agent.state_dict(),
        'item2id': item2id,
        'item_dim': item_dim,
        'hidden_dim': hidden_dim,
    }, path)
    print(f"PPOAgent saved to {path}")


def load_ppo_agent(path, device):
    ckpt = torch.load(path, map_location=device, weights_only=False)

    item2id = ckpt['item2id']
    item_dim = ckpt['item_dim']
    hidden_dim = ckpt['hidden_dim']

    agent = PPOAgent(item_dim=item_dim, hidden_dim=hidden_dim).to(device)
    agent.load_state_dict(ckpt['state_dict'])
    agent.eval()
    print(f"PPOAgent loaded from {path}  (item_dim={item_dim}, hidden={hidden_dim})")
    return agent, item2id


In [ ]:
reward_model, item2id = load_reward_model("/kaggle/input/datasets/ireneskvortsova/reward-model-all-saved/reward_model_ep10.pth", device)

RewardModel loaded from /kaggle/input/datasets/ireneskvortsova/reward-model-all-saved/reward_model_ep10.pth  (item_dim=1054859, hidden=128)


In [ ]:
def train_reward_model(dataset, reward_model, device, epochs=3, batch_size=1024, lr=3e-4):
    n_pos = sum(1 for _, _, _, l in dataset.samples if l == 1.0)
    n_neg = len(dataset) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], device=device)

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    optimizer = torch.optim.Adam(reward_model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    for epoch in range(epochs):
        reward_model.train()
        total_loss, total_acc = 0.0, 0.0
        total_tp, total_fp, total_fn = 0, 0, 0
        for batch in tqdm(loader, desc=f"RewardModel epoch {epoch+1}/{epochs}"):
            state = batch['state'].to(device)
            item = batch['item'].to(device)
            category_history = batch['category_history'].to(device)
            labels = batch['label'].to(device)

            logits = reward_model(state, item, category_history)
            loss = criterion(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(reward_model.parameters(), 1.0)
            optimizer.step()

            preds = (torch.sigmoid(logits) > 0.5).float()
            total_loss += loss.item()
            total_acc  += (preds == labels).float().mean().item()
            tp = ((preds == 1) & (labels == 1)).sum().item()
            fp = ((preds == 1) & (labels == 0)).sum().item()
            fn = ((preds == 0) & (labels == 1)).sum().item()
            total_tp += tp; total_fp += fp; total_fn += fn

        n = len(loader)
        precision = total_tp / max(total_tp + total_fp, 1)
        recall = total_tp / max(total_tp + total_fn, 1)
        print(f"  loss={total_loss/n:.4f}  acc={total_acc/n:.4f}  precision={precision:.4f}  recall={recall:.4f}")
        save_reward_model(reward_model, item2id, item_dim, num_categories, hidden_dim=128, path=f"reward_model_ep{epoch}.pth")
    return reward_model

In [22]:
item_dim = len(item2id)
item_dim

1308628

In [ ]:
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# item2id = slates_ds.get_item2id()
# item_dim = slates_ds.get_item_dim()
num_categories = 290 #int(itemattr['category'].max()) # 290
item_cat_tensor = torch.zeros(item_dim + 1, dtype=torch.long)
for original_id, remapped_id in item2id.items():
  item_cat_tensor[remapped_id] = int(itemattr['category'][original_id]) + 1  # +1: 0=PAD

In [26]:
item_cat_tensor.shape

torch.Size([1308629])

In [ ]:
import math

class SlateEnv:
      def __init__(self, reward_model, dat, item2id, item_cat_tensor,
                   max_slate_len=25, n_users=None, device='cuda'):
          self.reward_model = reward_model
          self.dat = dat
          self.item2id = item2id
          self.item_cat = item_cat_tensor
          self.max_slate_len = max_slate_len
          self.device = device
          self.n_users = n_users if n_users else dat['slate'].shape[0]
          self.n_slates = dat['slate'].shape[1]
          self.u = -1
          self.t = -1

      def reset(self):
          while True:
              u = random.randint(0, self.n_users - 1)
              t = random.randint(0, self.n_slates - 1)
              slate_raw = self.dat['slate'][u, t]
              valid_ids = [
                  self.item2id[s.item()]
                  for s in slate_raw
                  if s.item() >= 3 and s.item() in self.item2id
              ]
              if len(valid_ids) > 3:
                  self.u = u
                  self.t = t
                  break

          self.slate_len = len(valid_ids)
          self.slate_ids = valid_ids
          self.prefix = valid_ids[:3]
          self.shown_mask = [True, True, True] + [False] * (len(valid_ids) - 3)

          click_orig = self.dat['click'][u, t].item()
          if click_orig >= 3 and click_orig in self.item2id:
              self.click_id = self.item2id[click_orig]
          else:
              self.click_id = None

          return self._get_obs()

      def _pad(self, seq):
          res = torch.zeros(self.max_slate_len, dtype=torch.long)
          n = min(len(seq), self.max_slate_len)
          if n > 0:
              res[-n:] = torch.tensor(seq[-n:], dtype=torch.long)
          return res

      def _get_obs(self):
          candidates = [
              item_id if not shown else 0
              for item_id, shown in zip(self.slate_ids, self.shown_mask)
          ]

          padded_candidates = torch.zeros(self.max_slate_len, dtype=torch.long)
          n = len(candidates)
          padded_candidates[:n] = torch.tensor(candidates, dtype=torch.long)

          state = self._pad(self.prefix)
          return state, padded_candidates

      def step(self, action_idx):
          try:
              item_id = self.slate_ids[action_idx]
          except:
              print("self.slate_ids=", self.slate_ids)
              print("action_idx=", action_idx)
              print(f"u={self.u}, t={self.t}")

          state_t = self._pad(self.prefix).unsqueeze(0).to(self.device)
          item_t  = torch.tensor([item_id], dtype=torch.long, device=self.device)
          cat_seq = [self.item_cat[iid].item() for iid in self.prefix] + [self.item_cat[item_id].item()]
          cat_t   = self._pad(cat_seq).unsqueeze(0).to(self.device)
          p_cont  = self.reward_model.p_continue(state_t, cat_t, item_t).item()
          rank = len(self.prefix) - 2
          is_clicked = (self.click_id is not None) and (item_id == self.click_id)
          ndcg = (1.0 / math.log2(rank + 1)) if is_clicked else 0.0

          self.prefix.append(item_id)
          self.shown_mask[action_idx] = True

          all_shown = all(self.shown_mask)
          done = (random.random() > p_cont) or all_shown

          reward = p_cont / (self.slate_len - 1) + ndcg
          if done:
              return None, None, reward, True
          return *self._get_obs(), reward, False

In [ ]:
class PPOAgent(nn.Module):
  def __init__(self, item_dim, hidden_dim, padding_idx=0):
      super().__init__()
      self.padding_idx = padding_idx
      self.embedding = nn.Embedding(item_dim + 1, hidden_dim, padding_idx=padding_idx)
      self.encoder = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
      self.actor = nn.Sequential(
          nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 1)
      )
      self.critic = nn.Sequential(
          nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 1)
      )

  def get_state_repr(self, states_seq):
      embeds = self.embedding(states_seq) # [B, L, H]
      _, h_n = self.encoder(embeds)
      return h_n.squeeze(0)

  def _logits(self, state_seq, candidates):
      s = self.get_state_repr(state_seq) # [B, H]
      cand_emb = self.embedding(candidates) # [B, C, H]
      s_exp = s.unsqueeze(1).expand_as(cand_emb) # [B, C, H]
      logits = self.actor(torch.cat([s_exp, cand_emb], dim=-1)).squeeze(-1)
      logits = logits.masked_fill(candidates == self.padding_idx, float('-inf'))
      return logits

  def act(self, state_seq, candidates):
      dist = Categorical(logits=self._logits(state_seq, candidates))
      action = dist.sample()
      return action, dist.log_prob(action), self.critic(self.get_state_repr(state_seq)).squeeze(-1), dist.entropy()

  def evaluate(self, state_seq, candidates, actions):
      dist = Categorical(logits=self._logits(state_seq, candidates))
      return dist.log_prob(actions), self.critic(self.get_state_repr(state_seq)).squeeze(-1), dist.entropy()

In [ ]:
class RolloutBuffer:
  def __init__(self):
      self.states = []
      self.candidates = []
      self.actions = []
      self.log_probs = []
      self.values = []
      self.rewards = []
      self.dones = []

  def add(self, state, candidates, action, log_prob, value, reward, done):
      self.states.append(state)
      self.candidates.append(candidates)
      self.actions.append(action)
      self.log_probs.append(log_prob)
      self.values.append(value)
      self.rewards.append(reward)
      self.dones.append(done)

  def clear(self):
      self.__init__()

  def compute_gae(self, last_value, gamma=0.99, lam=0.95):
      n = len(self.rewards)
      advantages = torch.zeros(n)
      gae = 0.0
      for t in reversed(range(n)):
          next_val = 0.0 if self.dones[t] else (
              self.values[t + 1].item() if t + 1 < n else last_value
          )
          delta = self.rewards[t] + gamma * next_val - self.values[t].item()
          gae = delta + gamma * lam * (0.0 if self.dones[t] else gae)
          advantages[t] = gae
      returns = advantages + torch.tensor([v.item() for v in self.values])
      return returns, advantages

  def as_tensors(self, device):
      return (
          torch.stack(self.states).to(device),
          torch.stack(self.candidates).to(device),
          torch.stack(self.actions).to(device),
          torch.stack(self.log_probs).to(device),
      )

In [ ]:
def ppo_update(agent, optimizer_actor, optimizer_critic, buffer, last_value,
             gamma=0.99, lam=0.95, clip_eps=0.2,
             vf_coef=0.5, ent_coef=0.02,
             n_epochs=4, mini_batch_size=64, device='cuda'):

  returns, advantages = buffer.compute_gae(last_value, gamma, lam)
  advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

  states, candidates, actions, old_log_probs = buffer.as_tensors(device)
  returns = returns.to(device)
  advantages = advantages.to(device)

  stats = {'pg_loss': 0., 'vf_loss': 0., 'entropy': 0.}
  n_updates = 0
  n = len(returns)

  for _ in range(n_epochs):
      for idx in torch.randperm(n).split(mini_batch_size):
          new_log_probs, _, entropy = agent.evaluate(
              states[idx], candidates[idx], actions[idx]
          )
          ratio = torch.exp(new_log_probs - old_log_probs[idx])
          pg_loss = -torch.min(
              ratio * advantages[idx],
              torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * advantages[idx]
          ).mean()

          optimizer_actor.zero_grad()
          loss1 = pg_loss + ent_coef * (-entropy.mean())
          loss1.backward()
          torch.nn.utils.clip_grad_norm_(
              list(ppo_agent.actor.parameters()) +
              list(ppo_agent.encoder.parameters()) +
              list(ppo_agent.embedding.parameters()),
              0.5
          )
          optimizer_actor.step()
          _, new_values, _ = agent.evaluate(states[idx], candidates[idx], actions[idx])
          vf_loss = F.mse_loss(new_values, returns[idx])

          optimizer_critic.zero_grad()
          vf_loss.backward()
          torch.nn.utils.clip_grad_norm_(ppo_agent.critic.parameters(), 0.5)
          optimizer_critic.step()

          stats['pg_loss'] += pg_loss.item()
          stats['vf_loss'] += vf_loss.item()
          stats['entropy'] += entropy.mean().item()
          n_updates += 1

  return {k: v / n_updates for k, v in stats.items()}

In [ ]:
from IPython.display import clear_output
import matplotlib.pyplot as plt

def plot_training(ep_rewards, all_stats, steps_history):
    clear_output(wait=True)

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle('PPO Training', fontsize=14)
    ax = axes[0, 0]
    ax.plot(steps_history, ep_rewards, alpha=0.3, color='blue', label='raw')

    if len(ep_rewards) >= 25:
        smoothed = np.convolve(ep_rewards, np.ones(25)/25, mode='valid')
        ax.plot(steps_history[24:], smoothed, color='blue', linewidth=2, label='MA-20')

    ax.set_title('Episode Reward')
    ax.set_xlabel('Steps')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    if all_stats['policy_loss']:
        ax.plot(all_stats['policy_loss'], color='red')
    ax.set_title('Policy Loss')
    ax.set_xlabel('Updates')
    ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    if all_stats['value_loss']:
        ax.plot(all_stats['value_loss'], color='green')
    ax.set_title('Value Loss')
    ax.set_xlabel('Updates')
    ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    if all_stats['entropy']:
        ax.plot(all_stats['entropy'], color='orange')
    ax.set_title('Entropy')
    ax.set_xlabel('Updates')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


In [ ]:
def train_ppo(agent, env, optimizer_actor, optimizer_critic, device,
                n_steps=2048, total_steps=500_000,
                gamma=0.99, lam=0.95, clip_eps=0.2,
                vf_coef=0.5, ent_coef=0.02,
                n_epochs=2, mini_batch_size=128, plot_every=10):

    buffer = RolloutBuffer()
    ep_rewards, ep_reward = [], 0.0
    state, slate = env.reset()

    steps_done = 0
    steps_history = []

    all_stats = {'policy_loss': [], 'value_loss': [], 'entropy': []}

    while steps_done < total_steps:
      buffer.clear()

      for _ in range(n_steps):
          if state is None:
              ep_rewards.append(ep_reward)
              ep_reward = 0.0
              steps_history.append(steps_done)
              state, slate = env.reset()

          state_t = state.unsqueeze(0).to(device)
          slate_t = slate.unsqueeze(0).to(device)

          with torch.no_grad():
              action, log_prob, value, _ = agent.act(state_t, slate_t)

          next_state, next_slate, reward, done = env.step(action.item())
          buffer.add(state, slate, action.squeeze(),
                     log_prob.squeeze(), value.squeeze(), reward, done)

          ep_reward += reward
          steps_done += 1
          state, slate = next_state, next_slate

      last_value = 0.0
      if state is not None:
          with torch.no_grad():
              last_value = agent.critic(
                  agent.get_state_repr(state.unsqueeze(0).to(device))
              ).item()

      stats = ppo_update(agent, optimizer_actor, optimizer_critic, buffer, last_value,
                         gamma=gamma, lam=lam, clip_eps=clip_eps,
                         vf_coef=vf_coef, ent_coef=ent_coef,
                         n_epochs=n_epochs, mini_batch_size=mini_batch_size,
                         device=device)
      if steps_done > 50_000:
          ppo_agent.embedding.weight.requires_grad=True
      all_stats['policy_loss'].append(stats['pg_loss'])
      all_stats['value_loss'].append(stats['vf_loss'])
      all_stats['entropy'].append(stats['entropy'])
      plot_training(ep_rewards, all_stats, steps_history)

    return ep_reward

In [ ]:
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# item2id = slates_ds.get_item2id()
# item_dim = slates_ds.get_item_dim()

# reward_model = RewardModel(item_dim, hidden_dim=128).to(device)
# train_reward_model(reward_model, dat, item2id, device, n_users=50_000, epochs=3)

env = SlateEnv(reward_model, dat, item2id, item_cat_tensor, n_users=150_000, max_slate_len=25, device=device)

ppo_agent = PPOAgent(item_dim, hidden_dim=128).to(device)
ppo_agent.embedding.weight.data.copy_(reward_model.item_emb.weight.data)
ppo_agent.embedding.weight.requires_grad=False
optimizer_actor = torch.optim.Adam(
      list(ppo_agent.embedding.parameters()) +
      list(ppo_agent.encoder.parameters()) +
      list(ppo_agent.actor.parameters()),
      lr=2e-4
)
optimizer_critic = torch.optim.Adam(
  ppo_agent.critic.parameters(),
  lr=3e-4
)


In [ ]:
ep_rewards = train_ppo(ppo_agent, env, optimizer_actor, optimizer_critic, device,
                     n_steps=2048, ent_coef=0.02, total_steps=500_000)

In [ ]:
save_ppo_agent(ppo_agent, item2id, item_dim, 128, "ppo_agent_learns.pth")